# Tutorial 5: Advanced Agent Techniques and Real-World Applications

In this tutorial, we'll explore advanced agent techniques in LangChain and apply them to create a sophisticated AI-powered research assistant.

In [1]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_community.tools import Tool
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_text_splitters import CharacterTextSplitter
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

load_dotenv()

llm = ChatGroq(model_name='qwen/qwen3-32b')

# Use OllamaEmbeddings if available, otherwise FakeEmbeddings
try:
    embeddings = OllamaEmbeddings(model='all-minilm', base_url=os.getenv('OLLAMA_EMBEDDING_URL'))
    _ = embeddings.embed_query('test')  # probe connection
except Exception:
    from langchain_community.embeddings import FakeEmbeddings
    embeddings = FakeEmbeddings(size=384)
    print('Ollama not available — using FakeEmbeddings for demo')

print("Setup complete.")

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Ollama not available — using FakeEmbeddings for demo
Setup complete.


## 1. Creating a Custom Agent with Specialized Capabilities

In [2]:
# In LangChain 1.0, create_react_agent handles the ReAct template and output parsing
# internally — no need for StringPromptTemplate or AgentOutputParser.
#
# The system_prompt parameter lets you customise agent behaviour:
system_prompt = (
    "You are an AI research assistant for scientific literature analysis.\n"
    "When answering:\n"
    "1. Search for relevant papers using VectorStoreSearch\n"
    "2. Summarize key findings using Summarize\n"
    "3. Provide analytical insights using Analyze\n"
    "Be thorough and cite specific findings from the papers."
)

print("System prompt defined.")

System prompt defined.


## 2. Implementing a """Multi-Agent""" System

In [3]:
# Load and process research papers
loader = DirectoryLoader('research_papers', glob='*.txt', loader_cls=TextLoader)
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)
vectorstore = FAISS.from_documents(texts, embeddings)
print(f'Loaded {len(texts)} document chunks')

# Define tools using @tool decorator (modern approach)
@tool
def vector_store_search(query: str) -> str:
    """Search research papers for relevant information."""
    results = vectorstore.similarity_search(query, k=2)
    return '\n'.join(d.page_content[:200] for d in results)

@tool
def summarize(text: str) -> str:
    """Summarize a given text concisely."""
    return llm.invoke([HumanMessage(content=f'Summarize in 3 sentences: {text[:500]}')]).content

@tool
def analyze(text: str) -> str:
    """Analyze text and provide key insights."""
    return llm.invoke([HumanMessage(content=f'Give 3 key insights from: {text[:500]}')]).content

tools = [vector_store_search, summarize, analyze]

# create_react_agent replaces LLMSingleActionAgent + AgentExecutor
research_agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt=system_prompt
)

# Test
result = research_agent.invoke({
    'messages': [HumanMessage(content='What renewable energy technologies are discussed? Summarize the key findings.')]
})
print(result["messages"][-1].content[:400])

Loaded 9 document chunks
The paper highlights advancements in three key renewable energy technologies:  
1. **Solar Photovoltaics (PV)**: Improved efficiency in solar panels and reduced costs, enabling broader adoption.  
2. **Wind Energy**: Expansion of offshore wind farms and advancements in turbine design for higher energy output.  
3. **Energy Storage**: Innovations in battery technology (e.g., lithium-ion, solid-stat


## 3. Developing a Context-Aware Agent with Long-Term Memory

In [4]:
# Memory in modern LangChain: MemorySaver checkpointer on the LangGraph agent
# This replaces ConversationBufferMemory + ConversationChain

memory_agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt=system_prompt,
    checkpointer=MemorySaver()
)

config = {'configurable': {'thread_id': 'research-session-1'}}

# Turn 1: ask about climate change
r1 = memory_agent.invoke(
    {'messages': [HumanMessage(content='What methods were used in the climate change study?')]},
    config
)
print('Turn 1:', r1['messages'][-1].content[:200])

# Turn 2: follow-up — agent remembers the context
r2 = memory_agent.invoke(
    {'messages': [HumanMessage(content='What were the main findings of that study?')]},
    config
)
print("Turn 2:", r2["messages"][-1].content[:200])

Turn 1: The climate change study utilized several methods, as summarized from the reviewed literature:

1. **Systematic Review of Peer-Reviewed Articles**: A comprehensive analysis of studies published betwee
Turn 2: The search results for the main findings appear to reference a different paper focused on **AI applications in healthcare** rather than climate change. However, based on the earlier context about the 


## 4. Building a Real-World Application: AI-Powered Research Assistant

In [5]:
def research_assistant(query: str, cfg: dict) -> str:
    response = memory_agent.invoke(
        {'messages': [HumanMessage(content=query)]}, cfg
    )
    answer = response['messages'][-1].content
    print(f'Q: {query}')
    print(f'A: {answer[:200]}\n')
    return answer

research_config = {'configurable': {'thread_id': 'research-queries'}}

research_assistant('What are the findings about coral reefs?', research_config)
research_assistant("What AI applications in medical imaging are mentioned?", research_config)

Q: What are the findings about coral reefs?
A: The search results retrieved information about artificial intelligence in healthcare, which appears unrelated to coral reef findings. This may indicate a mismatch in the search process. Would you like

Q: What AI applications in medical imaging are mentioned?
A: The tool response you referenced contains incomplete text (cut off mid-sentence) and appears to be a placeholder or error, as it mentions "artificial intelligence in healthcare" but lacks specific det



'The tool response you referenced contains incomplete text (cut off mid-sentence) and appears to be a placeholder or error, as it mentions "artificial intelligence in healthcare" but lacks specific details about **medical imaging**. The provided snippet does not include the actual findings or applications discussed in the paper. \n\nTo address your question accurately, I would need access to the full paper or a proper summary of its contents. If you\'d like, I can:\n1. Refine the search for **AI applications in medical imaging** using updated terms.\n2. Explore **coral reef studies** further if you\'d prefer to return to that topic.\n\nLet me know how you\'d like to proceed!'

## Summary and Key Takeaways

In this tutorial, we've explored advanced agent techniques in LangChain and applied them to create a sophisticated AI-powered research assistant:

1. **Custom Agent Creation**: We developed a specialized agent for scientific literature analysis, demonstrating how to tailor agents for specific domains.

2. **Multi-Agent System**: By combining multiple tools (search, summarize, analyze), we created a versatile system capable of handling complex research tasks.

3. **Context-Aware Agent with Long-Term Memory**: We implemented a conversation memory, allowing the agent to maintain context across multiple interactions and provide more coherent and relevant responses over time.

4. **Real-World Application**: We built an AI-powered research assistant that can analyze scientific papers, extract key information, and provide insights on complex topics like climate change.

Key Takeaways:
- Advanced agents can be tailored for specific domains and tasks, greatly enhancing their effectiveness.
- Combining multiple tools and agent types allows for the creation of powerful, multi-functional AI systems.
- Implementing memory and context awareness significantly improves the quality and coherence of agent responses over time.
- Real-world applications of these techniques can lead to powerful AI assistants capable of handling complex, domain-specific tasks.

Next Steps:
- Experiment with different combinations of tools and agent types for your specific use cases.
- Explore ways to further enhance the agent's memory and context understanding capabilities.
- Consider integrating external APIs or databases to expand the agent's knowledge and capabilities.
